# CivicLens AI — Streetlight fine-tune (one-click Colab)

Fine-tunes a **YOLOv8n** streetlight detector on a Roboflow dataset and deploys the trained weights back to Roboflow's hosted inference API — the same endpoint the CivicLens backend already calls.

**Prerequisites (5 min setup):**

1. Free Roboflow account → https://app.roboflow.com  
2. Grab your **Private API key** → https://app.roboflow.com/settings/api  
3. Fork one of the streetlight datasets into your workspace:  
   - Recommended: https://universe.roboflow.com/pothole-fw8hn/street_light-0wrmn (964 imgs, Working/Nonworking/Flicker)  
   - Alt: https://universe.roboflow.com/godspeed-yqpeo/damaged-lights (619 imgs, Working/Not-Working)  
4. In your forked project → **Versions → Generate New Version** (accept defaults).

**Then:** Runtime → Change runtime type → **T4 GPU** → Runtime → **Run all**.  

⏱ Total wall time on Colab T4: **~5-8 min** for 60 epochs.

## 1 · Install dependencies (~30 s)

In [ ]:
%pip install -q roboflow ultralytics

## 2 · Configure — **edit these four values**

- `ROBOFLOW_API_KEY` — from https://app.roboflow.com/settings/api  
- `ROBOFLOW_WORKSPACE` — the slug in your Roboflow URL, e.g. `dev-dipeshkumar`  
- `ROBOFLOW_PROJECT` — the slug of your forked streetlight project, e.g. `civiclens-streetlight`  
- `ROBOFLOW_DATASET_VERSION` — the version number you generated (usually `1`)

In [ ]:
import os

# --- FILL THESE IN ---
os.environ['ROBOFLOW_API_KEY']        = 'rf_paste_your_key_here'
os.environ['ROBOFLOW_WORKSPACE']      = 'your-workspace-slug'
os.environ['ROBOFLOW_PROJECT']        = 'civiclens-streetlight'
os.environ['ROBOFLOW_DATASET_VERSION']= '1'

# --- Training hyperparameters (safe defaults) ---
EPOCHS   = 60
IMG_SIZE = 640
BATCH    = 16
ARCH     = 'yolov8n.pt'   # nano — fastest. Swap for yolov8s.pt if you want more accuracy.

assert os.environ['ROBOFLOW_API_KEY'].startswith('rf_') or len(os.environ['ROBOFLOW_API_KEY']) > 10, \
    'Paste your Roboflow API key first.'
print('Config looks good ✅')

## 3 · Download the dataset from Roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
project = rf.workspace(os.environ['ROBOFLOW_WORKSPACE']).project(os.environ['ROBOFLOW_PROJECT'])
version = project.version(int(os.environ['ROBOFLOW_DATASET_VERSION']))
dataset = version.download('yolov8')

data_yaml = f'{dataset.location}/data.yaml'
print(f'\n✅ Dataset ready at: {data_yaml}')
!head -20 $data_yaml

## 4 · Verify GPU is available (optional but recommended)

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU for a ~10× speedup.')

## 5 · Fine-tune YOLOv8 on your dataset (~4–7 min on T4)

In [ ]:
from ultralytics import YOLO

model = YOLO(ARCH)
results = model.train(
    data=data_yaml,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    project='runs',
    name='streetlight',
    exist_ok=True,
    patience=15,     # early-stop if no improvement
    plots=True,
)

print('\n✅ Training complete.')
print(f'Best weights: {results.save_dir}/weights/best.pt')

## 6 · Evaluate — how good is it?

In [ ]:
metrics = model.val(data=data_yaml)
print(f'\n📊 mAP@50     : {metrics.box.map50:.3f}   ({"good" if metrics.box.map50 > 0.55 else "ok" if metrics.box.map50 > 0.35 else "weak"})')
print(f'📊 mAP@50-95  : {metrics.box.map:.3f}')
print(f'📊 Precision  : {metrics.box.mp:.3f}')
print(f'📊 Recall     : {metrics.box.mr:.3f}')

### Optional: try the model on a test image from the dataset

In [ ]:
import glob, os, IPython.display as d
test_imgs = glob.glob(f'{dataset.location}/test/images/*.jpg')[:3] or \
            glob.glob(f'{dataset.location}/valid/images/*.jpg')[:3]
if test_imgs:
    preds = model.predict(test_imgs, conf=0.25, save=True)
    out_dir = preds[0].save_dir
    for f in sorted(os.listdir(out_dir))[:3]:
        d.display(d.Image(f'{out_dir}/{f}'))
else:
    print('No test images found; skipping preview.')

## 7 · Deploy to Roboflow hosted inference

This uploads your `best.pt` to the same Roboflow project version. After this cell finishes, the model is live at:  
`https://serverless.roboflow.com/<project>/<version>`  
— which is exactly what the CivicLens backend calls.

In [ ]:
version.deploy(
    model_type='yolov8',
    model_path='runs/streetlight',
    filename='weights/best.pt',
)

slug = f"{os.environ['ROBOFLOW_PROJECT']}/{os.environ['ROBOFLOW_DATASET_VERSION']}"
print('\n' + '=' * 70)
print(f'  ✅  Model deployed! Slug: {slug}')
print(f'  ▶️  Append to your backend .env:')
print()
print(f'  ROBOFLOW_MODELS=pothole:pothole-detection-yolov8/1,\\')
print(f'                  garbage:garbage_detection-wvzwv/9,\\')
print(f'                  road_damage:road_defects-eif9i/9,\\')
print(f'                  streetlight:{slug}')
print('=' * 70)

## 8 · Smoke test the deployed model (proves it's live on Roboflow's servers)

In [ ]:
import base64, requests, json

with open(test_imgs[0], 'rb') as f:
    b64 = base64.b64encode(f.read()).decode()

url = f"https://serverless.roboflow.com/{slug}?api_key={os.environ['ROBOFLOW_API_KEY']}"
r = requests.post(url, data=b64, headers={'Content-Type': 'application/x-www-form-urlencoded'})
print('HTTP', r.status_code)
print(json.dumps(r.json(), indent=2)[:800])

## 9 · Download your `best.pt` (optional backup)

In [ ]:
from google.colab import files
files.download('runs/streetlight/weights/best.pt')

---

## What just happened

1. Pulled your labeled Roboflow dataset in YOLOv8 format.  
2. Fine-tuned YOLOv8-nano on top of COCO-pretrained weights for `EPOCHS` epochs.  
3. Validated and printed mAP.  
4. Uploaded the best checkpoint back to Roboflow hosted inference.  
5. Verified the deployed endpoint responds.  

## What to do next

Copy the `ROBOFLOW_MODELS=...` line printed by cell 7 into your CivicLens backend's `.env`, restart uvicorn, and every streetlight photo uploaded through the citizen wizard will now flow through your fine-tuned model — logged as:

```
ROBOFLOW civiclens-streetlight -> streetlight (Nonworking) conf=0.87 area=6.3%
```

The severity engine automatically routes any `streetlight` detection to the **Lighting** department — no frontend changes required.  

If mAP@50 came out low (< 0.4), try:  
- Increasing `EPOCHS` to 100–150  
- Switching `ARCH` to `'yolov8s.pt'` (small, more accurate, ~2× training time)  
- Generating a new dataset version in Roboflow with more augmentations (rotation, brightness, saturation)